# Propiedades de mezcla gaseosa y coeficientes de transporte

Este notebook verifica el cálculo de **propiedades termofísicas de mezcla**, **coeficientes de difusión** y **coeficientes de transferencia de calor y masa** en un lecho granular usando la nueva arquitectura modular.

## Módulos utilizados

| Módulo | Función | Responsabilidad |
|---|---|---|
| `src.physics.thermodynamics.pure_gas` | `build_pure_gas_properties` | Propiedades puras por especie |
| `src.physics.mixture_gas` | `compute_gas_mixture_properties` | Propiedades de mezcla y difusión molecular |
| `src.physics.transport.diffusion` | `knudsen_diffusivity`, `pore_diffusivity`, `effective_diffusivity` | Difusión en poros |
| `src.physics.transport.transfer_coefficients` | `compute_transfer_coefficients` | Números adimensionales y coeficientes convectivos |

## Convención de arrays

Todas las funciones de mezcla usan **node-first**: `x` tiene shape `(N, nc)`, donde `N` es el número de nodos y `nc` el número de especies. Los resultados por especie tienen shape `(N, nc)`.

## Configuración del test

- **4 nodos**, presión constante de 9 bar
- Temperaturas: 300, 320, 330, 350 K
- Mezcla binaria **CH4 / CO2** con gradiente axial de composición (de 70/30 a 40/60)
- N2 presente como trazador nulo (x = 0) para verificar estabilidad numérica con especies a dilución nula
- Lecho granular con porosidad y tortuosidad variables entre nodos

In [2]:
import os, sys
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.physics.thermodynamics.pure_gas import build_pure_gas_properties
from src.physics.mixture_gas import compute_gas_mixture_properties
from src.physics.transport.diffusion import (
    knudsen_diffusivity,
    pore_diffusivity,
    effective_diffusivity,
)
from src.physics.transport.transfer_coefficients import compute_transfer_coefficients
from src.units.adsorber.config.transport import build_transport_config

DB_PATH = os.path.join(ROOT, "materials", "fluids", "gasdb.txt")
R_GAS   = 8.31446261815324   # J/mol/K

print("Imports OK")

Imports OK


## Definición de condiciones del test

In [3]:
SPECIES = ["CH4", "CO2", "N2"]
nc = len(SPECIES)
N  = 4

# --- Gas puro (modo polynomial) ---
prop_gas = build_pure_gas_properties(
    species=SPECIES, mode="polynomial", db_path=DB_PATH
)

# --- Condiciones termodinámicas ---
P_Pa = np.full(N, 9e5, dtype=float)                          # [Pa]
Tg   = np.array([300., 320., 330., 350.], dtype=float)       # [K]
Ts   = Tg.copy()                                             # [K] — igual al gas en este test

# --- Composición: shape (N, nc), node-first ---
x = np.array(
    [[0.7, 0.3, 0.0],
     [0.6, 0.4, 0.0],
     [0.5, 0.5, 0.0],
     [0.4, 0.6, 0.0]],
    dtype=float,
)

# --- Propiedades del lecho granular ---
prop_lecho = {
    "r_pore": np.array([2e-6, 2e-6, 1e-6, 1e-6], dtype=float),   # [m]
    "eps":    np.array([0.40, 0.40, 0.35, 0.35], dtype=float),    # [-]
    "tau":    np.array([3.0,  3.0,  3.5,  3.5 ], dtype=float),    # [-]
    "D_p":    np.full(N, 5e-3, dtype=float),                       # [m]
    "a_surf": np.full(N, 6.0*(1-0.4)/5e-3, dtype=float),          # [m²/m³]
}

# --- Condiciones de flujo ---
u_rel = np.array([0.20, 0.25, 0.30, 0.35], dtype=float)          # [m/s]
Di    = 0.10                                                       # [m] diámetro interno

IDX = [f"node_{i}" for i in range(N)]

print("Setup OK")

Setup OK


## TEST 1 — Propiedades de mezcla

`compute_gas_mixture_properties` aplica la regla de Wilke para viscosidad y conductividad,
promedios molares para `Cp` y `h`, y la ecuación de Chapman–Enskog + Stefan–Maxwell para
la difusión molecular `Dim`.

In [4]:
gp = compute_gas_mixture_properties(
    P_Pa=P_Pa, Tg=Tg, x=x,
    prop_gas=prop_gas, n_comp=nc, N=N,
)

# Propiedades derivadas (gas ideal)
# h_mix = sum_i(x_i * h_i)  — ya calculado internamente como h_i(N,nc) · x
h_molar_mix = np.sum(gp["h_i"] * x, axis=1)        # (N,) [J/mol]
h_mass_mix  = h_molar_mix / gp["MWmix"]             # (N,) [J/kg]
# u = h - RT (gas ideal, referencia = Tref → no se resta pv0 porque h ya es relativa a Tref)
Tref_mean = np.mean([prop_gas["Tref"][i] for i in range(nc)])  # aprox para u derivada
u_molar_mix = h_molar_mix - R_GAS * (Tg - Tref_mean)
u_mass_mix  = u_molar_mix / gp["MWmix"]

df_mix = pd.DataFrame(
    {
        "P [Pa]":             P_Pa,
        "T [K]":              Tg,
        "MWmix [kg/mol]":     gp["MWmix"],
        "rho [kg/m³]":        gp["rho"],
        "mu [Pa·s]":          gp["mu"],
        "k [W/m/K]":          gp["k"],
        "Cp_mass [J/kg/K]":   gp["Cp_mass"],
        "Cp_molar [J/mol/K]": gp["Cp_molar"],
        "u_mass [J/kg]":      u_mass_mix,
        "u_molar [J/mol]":    u_molar_mix,
        "h_mass [J/kg]":      h_mass_mix,
        "h_molar [J/mol]":    h_molar_mix,
    },
    index=IDX,
)

pd.set_option("display.float_format", "{:.6g}".format)
print("=== Propiedades de mezcla ===\n")
display(df_mix)

=== Propiedades de mezcla ===



,P [Pa],T [K],MWmix [kg/mol],rho [kg/m³],mu [Pa·s],k [W/m/K],Cp_mass [J/kg/K],Cp_molar [J/mol/K],u_mass [J/kg],u_molar [J/mol],h_mass [J/kg],h_molar [J/mol]
node_0,900000,300,0.0244326,8.8157,1.30168e-05,0.0250488,1467.65,35.8584,693562,16945.5,694242,16962.1
node_1,900000,320,0.0272293,9.21075,1.42548e-05,0.025234,1361.74,37.0791,670840,18266.5,677558,18449.4
node_2,900000,330,0.030026,9.84899,1.5079e-05,0.0245618,1257.03,37.7437,643712,19328.1,652573,19594.2
node_3,900000,350,0.0328227,10.1511,1.62657e-05,0.0250152,1183.63,38.85,630703,20701.4,643875,21133.7


## TEST 2 — Composición y difusión molecular en mezcla (`Dim`)

`Dim` es la difusividad de cada especie en la mezcla multicomponente (Chapman–Enskog + Stefan–Maxwell).
Shape: `(N, nc)` — node-first.

In [5]:
df_x = pd.DataFrame(x, index=IDX, columns=SPECIES)
print("=== Fracciones molares ===\n")
display(df_x)

df_Dim = pd.DataFrame(gp["Dim"], index=IDX, columns=SPECIES)
print("\n=== Dim — difusión molecular en mezcla [m²/s] ===\n")
display(df_Dim)

=== Fracciones molares ===



,CH4,CO2,N2
node_0,0.7,0.3,0
node_1,0.6,0.4,0
node_2,0.5,0.5,0
node_3,0.4,0.6,0



=== Dim — difusión molecular en mezcla [m²/s] ===



,CH4,CO2,N2
node_0,6.32394e-06,2.71026e-06,2.20728e-06
node_1,5.3477e-06,3.56513e-06,2.38909e-06
node_2,4.52863e-06,4.52863e-06,2.43847e-06
node_3,4.2052e-06,6.3078e-06,2.61809e-06


## TEST 3 — Difusión en poros (`Dkn`, `Dpore`, `Deff`)

- `Dkn` — difusión de Knudsen: limitada por colisiones con las paredes del poro
- `Dpore` — difusión combinada (Bosanquet): `1/Dpore = 1/Dim + 1/Dkn`
- `Deff` — difusión efectiva en el lecho: `Deff = (eps/tau) * Dpore`

In [6]:
Dkn   = knudsen_diffusivity(T=Tg, MW=prop_gas["MW"], r_pore=prop_lecho["r_pore"])
Dpore = pore_diffusivity(Dim=gp["Dim"], Dkn=Dkn)
Deff  = effective_diffusivity(Dpor=Dpore, eps=prop_lecho["eps"], tau=prop_lecho["tau"])

df_Dkn   = pd.DataFrame(Dkn,   index=IDX, columns=SPECIES)
df_Dpore = pd.DataFrame(Dpore, index=IDX, columns=SPECIES)
df_Deff  = pd.DataFrame(Deff,  index=IDX, columns=SPECIES)

print("=== Dkn — difusión de Knudsen [m²/s] ===\n")
display(df_Dkn)

print("\n=== Dpore — difusión combinada Bosanquet [m²/s] ===\n")
display(df_Dpore)

print("\n=== Deff — difusión efectiva en lecho [m²/s] ===\n")
display(df_Deff)

=== Dkn — difusión de Knudsen [m²/s] ===



,CH4,CO2,N2
node_0,0.00083898,0.000506541,0.000634894
node_1,0.000866495,0.000523153,0.000655716
node_2,0.000439965,0.000265632,0.000332941
node_3,0.000453101,0.000273563,0.000342882



=== Dpore — difusión combinada Bosanquet [m²/s] ===



,CH4,CO2,N2
node_0,6.27663e-06,2.69584e-06,2.19963e-06
node_1,5.3149e-06,3.541e-06,2.38042e-06
node_2,4.4825e-06,4.45272e-06,2.42074e-06
node_3,4.16653e-06,6.16564e-06,2.59825e-06



=== Deff — difusión efectiva en lecho [m²/s] ===



,CH4,CO2,N2
node_0,8.36884e-07,3.59445e-07,2.93285e-07
node_1,7.08653e-07,4.72134e-07,3.17389e-07
node_2,4.4825e-07,4.45272e-07,2.42074e-07
node_3,4.16653e-07,6.16564e-07,2.59825e-07


## TEST 4 — Números adimensionales y coeficientes convectivos

`compute_transfer_coefficients` calcula en modo `"correlation"`:

- **Re** — Reynolds basado en `D_p` y `u_rel`
- **Pr** — Prandtl de la mezcla
- **Sc** — Schmidt por especie
- **Nu_bed** — Nusselt gas–partícula: `Nu = 2 + 1.1·Re^0.6·Pr^(1/3)`
- **Nu_wall** — Nusselt gas–pared: correlación basada en `Di`
- **Sh** — Sherwood por especie: `Sh = 2 + 1.1·Re^0.6·Sc^(1/3)`
- **beta_mt** — coeficiente de transferencia de masa gas–partícula [m/s]
- **h_bed** — coeficiente convectivo gas–partícula [W/m²/K]
- **h_wall** — coeficiente convectivo gas–pared [W/m²/K]

Las propiedades de transporte se evalúan a temperatura de película `Tfilm = 0.5·(Tg + Ts)`.

In [7]:
trans_cfg = build_transport_config(mode="correlation", n_comp=nc, N=N)

tp = compute_transfer_coefficients(
    Tg=Tg, Ts=Ts, x=x,
    gas_props=gp, u_rel=u_rel,
    prop_gas=prop_gas, prop_lecho=prop_lecho,
    Di=Di, trans_config=trans_cfg,
    n_comp=nc, N=N,
)

# --- tabla Pr / Re / Nu / h ---
df_conv = pd.DataFrame(
    {
        "Tfilm [K]":       tp["Tfilm"],
        "Pr [-]":          tp["Pr"],
        "Re [-]":          tp["Re"],
        "Nu_bed [-]":      tp["Nu_bed"],
        "h_bed [W/m²/K]": tp["h_bed"],
        "Nu_wall [-]":     tp["Nu_wall"],
        "h_wall [W/m²/K]":tp["h_wall"],
    },
    index=IDX,
)
print("=== Pr, Re, Nu y coeficientes convectivos ===\n")
display(df_conv)

# --- Sc, Sh, beta_mt por especie ---
# tp["Sc"] y tp["Sh"] tienen shape (N, nc) o (nc, N) — verificar
Sc_arr     = np.asarray(tp["Sc"])
Sh_arr     = np.asarray(tp["Sh"])
beta_arr   = np.asarray(tp["beta_mt"])

# Normalizar a (N, nc) si vienen (nc, N)
if Sc_arr.shape == (nc, N):
    Sc_arr, Sh_arr, beta_arr = Sc_arr.T, Sh_arr.T, beta_arr.T

df_Sc   = pd.DataFrame(Sc_arr,   index=IDX, columns=SPECIES)
df_Sh   = pd.DataFrame(Sh_arr,   index=IDX, columns=SPECIES)
df_beta = pd.DataFrame(beta_arr, index=IDX, columns=SPECIES)

print("\n=== Sc — Schmidt por especie ===\n")
display(df_Sc)

print("\n=== Sh — Sherwood por especie ===\n")
display(df_Sh)

print("\n=== beta_mt — coef. transferencia de masa [m/s] ===\n")
display(df_beta)

=== Pr, Re, Nu y coeficientes convectivos ===



,Tfilm [K],Pr [-],Re [-],Nu_bed [-],h_bed [W/m²/K],Nu_wall [-],h_wall [W/m²/K]
node_0,300,0.762675,677.253,52.1907,261.463,41.6954,10.4442
node_1,320,0.769251,807.69,57.9451,292.437,48.1698,12.1551
node_2,330,0.77172,979.738,64.8847,318.737,56.289,13.8256
node_3,350,0.769639,1092.14,69.0588,345.503,61.3324,15.3424



=== Sc — Schmidt por especie ===



,CH4,CO2,N2
node_0,0.233486,0.544801,0.668946
node_1,0.2894,0.4341,0.647788
node_2,0.338076,0.338076,0.627861
node_3,0.381042,0.254028,0.612033



=== Sh — Sherwood por especie ===



,CH4,CO2,N2
node_0,35.827,46.8665,50.0441
node_1,42.3867,48.2313,54.8304
node_2,49.7599,49.7599,60.7056
node_3,55.0498,48.3433,64.1277



=== beta_mt — coef. transferencia de masa [m/s] ===



,CH4,CO2,N2
node_0,0.0453136,0.0254041,0.0220923
node_1,0.0453343,0.0343902,0.0261989
node_2,0.0450689,0.0450689,0.0296058
node_3,0.0462991,0.060988,0.0335784


## TEST 5 — Dispersión axial y coeficiente de transferencia de masa LDF (`D_disp`, `k_mtc`)

Campos adicionales devueltos por `compute_transfer_coefficients` que alimentan directamente el RHS del modelo de columna PSA:

- `D_disp` — dispersión axial por especie `(nc, N)` [m²/s]
- `k_mtc` — coeficiente LDF gas–sólido por especie `(nc, N)` [1/s]

In [8]:
# k_mtc y D_disp tienen shape (nc, N) — transponemos para display
df_Ddisp = pd.DataFrame(tp["D_disp"].T, index=IDX, columns=SPECIES)
df_kmtc  = pd.DataFrame(tp["k_mtc"].T,  index=IDX, columns=SPECIES)

print("=== D_disp — dispersión axial [m²/s] ===\n")
display(df_Ddisp)

print("\n=== k_mtc — coef. transferencia de masa LDF [1/s] ===\n")
display(df_kmtc)

=== D_disp — dispersión axial [m²/s] ===



,CH4,CO2,N2
node_0,0.0005,0.0005,0.0005
node_1,0.000625,0.000625,0.000625
node_2,0.00075,0.00075,0.00075
node_3,0.000875,0.000875,0.000875



=== k_mtc — coef. transferencia de masa LDF [1/s] ===



,CH4,CO2,N2
node_0,32.6258,18.291,15.9065
node_1,32.6407,24.7609,18.8632
node_2,32.4496,32.4496,21.3162
node_3,33.3354,43.9114,24.1764
